# TP 2 : Sécurisation de la gestion de pharmacie (Spring Security + MySQL)

**Module :** Spring Boot - Microservices  **Prof :** Youness IDRISSI KHAMLICHI  **ENSA Fès**

Dans ce notebook je construis l'application `pharma_security` : authentification par formulaire, autorisation par rôle, mots de passe chiffrés (BCrypt) et utilisateurs stockés dans MySQL. Chaque cellule `%%writefile` crée un fichier du projet Maven, puis je lance l'application et je teste les accès de chaque rôle.

| Rôle | Accès autorisé |
|---|---|
| ADMIN | Gérer les utilisateurs, les médicaments, les stocks |
| PHARMACIEN | Consulter et modifier les stocks, vendre des médicaments |
| CAISSIER | Enregistrer les ventes uniquement |
| CLIENT | Consulter les médicaments disponibles (lecture seule) |



## 1. Structure du projet

In [1]:
import os
dossiers = [
    "pharma_security/src/main/java/ma/ensa/pharmasecurity/config",
    "pharma_security/src/main/java/ma/ensa/pharmasecurity/model",
    "pharma_security/src/main/java/ma/ensa/pharmasecurity/repository",
    "pharma_security/src/main/java/ma/ensa/pharmasecurity/controller",
    "pharma_security/src/main/java/ma/ensa/pharmasecurity/service",
    "pharma_security/src/main/resources/templates",
]
for d in dossiers:
    os.makedirs(d, exist_ok=True)
print("Dossiers créés :", len(dossiers))

Dossiers créés : 6


## 2. `pom.xml`
Par rapport au TP1, j'ajoute `spring-boot-starter-security` et `thymeleaf-extras-springsecurity6` (pour `sec:authorize` dans les pages).

In [2]:
%%writefile pharma_security/pom.xml
<?xml version="1.0" encoding="UTF-8"?>
<project xmlns="http://maven.apache.org/POM/4.0.0"
         xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"
         xsi:schemaLocation="http://maven.apache.org/POM/4.0.0 https://maven.apache.org/xsd/maven-4.0.0.xsd">
    <modelVersion>4.0.0</modelVersion>

    <parent>
        <groupId>org.springframework.boot</groupId>
        <artifactId>spring-boot-starter-parent</artifactId>
        <version>3.3.5</version>
        <relativePath/>
    </parent>

    <groupId>ma.ensa</groupId>
    <artifactId>pharma_security</artifactId>
    <version>0.0.1-SNAPSHOT</version>
    <name>pharma_security</name>

    <properties>
        <java.version>17</java.version>
    </properties>

    <dependencies>
        <dependency>
            <groupId>org.springframework.boot</groupId>
            <artifactId>spring-boot-starter-web</artifactId>
        </dependency>
        <dependency>
            <groupId>org.springframework.boot</groupId>
            <artifactId>spring-boot-starter-security</artifactId>
        </dependency>
        <dependency>
            <groupId>org.springframework.boot</groupId>
            <artifactId>spring-boot-starter-data-jpa</artifactId>
        </dependency>
        <dependency>
            <groupId>org.springframework.boot</groupId>
            <artifactId>spring-boot-starter-thymeleaf</artifactId>
        </dependency>
        <dependency>
            <groupId>org.thymeleaf.extras</groupId>
            <artifactId>thymeleaf-extras-springsecurity6</artifactId>
        </dependency>
        <dependency>
            <groupId>com.mysql</groupId>
            <artifactId>mysql-connector-j</artifactId>
            <scope>runtime</scope>
        </dependency>
        <dependency>
            <groupId>org.projectlombok</groupId>
            <artifactId>lombok</artifactId>
            <optional>true</optional>
        </dependency>
    </dependencies>

    <build>
        <plugins>
            <plugin>
                <groupId>org.springframework.boot</groupId>
                <artifactId>spring-boot-maven-plugin</artifactId>
                <configuration>
                    <excludes>
                        <exclude>
                            <groupId>org.projectlombok</groupId>
                            <artifactId>lombok</artifactId>
                        </exclude>
                    </excludes>
                </configuration>
            </plugin>
        </plugins>
    </build>
</project>

Writing pharma_security/pom.xml


## 3. Classe principale et configuration MySQL

In [3]:
%%writefile pharma_security/src/main/java/ma/ensa/pharmasecurity/PharmaSecurityApplication.java

package ma.ensa.pharmasecurity;

import org.springframework.boot.SpringApplication;
import org.springframework.boot.autoconfigure.SpringBootApplication;

@SpringBootApplication
public class PharmaSecurityApplication {

    public static void main(String[] args) {
        SpringApplication.run(PharmaSecurityApplication.class, args);
    }
}

Writing pharma_security/src/main/java/ma/ensa/pharmasecurity/PharmaSecurityApplication.java


In [4]:
%%writefile pharma_security/src/main/resources/application.properties

spring.application.name=pharma_security

# --- MySQL (la base est créée automatiquement) ---
spring.datasource.url=jdbc:mysql://localhost:3306/pharma_security_db?createDatabaseIfNotExist=true&useSSL=false&allowPublicKeyRetrieval=true&serverTimezone=UTC
spring.datasource.username=root
spring.datasource.password=

# --- JPA / Hibernate ---
spring.jpa.hibernate.ddl-auto=update
spring.jpa.show-sql=false

# 8081 pour ne pas entrer en conflit avec l'application du TP1
server.port=8081

Writing pharma_security/src/main/resources/application.properties


## 4. Modèle (entités JPA)

- `User` et `Role` : relation `@ManyToMany` (table `user_roles`). Le mot de passe est stocké **chiffré** (BCrypt), jamais en clair. La table s'appelle `users` pour éviter le mot réservé `user`.
- `Medicament` : nom, prix, quantité en stock, date d'expiration.
- `Vente` : trace de chaque vente (médicament, quantité, total, date, vendeur). Elle sert à justifier le rôle CAISSIER (« enregistrer les ventes »).

In [5]:
%%writefile pharma_security/src/main/java/ma/ensa/pharmasecurity/model/Role.java

package ma.ensa.pharmasecurity.model;

import jakarta.persistence.*;
import lombok.Getter;
import lombok.NoArgsConstructor;
import lombok.Setter;

@Entity
@Getter
@Setter
@NoArgsConstructor
public class Role {

    @Id
    @GeneratedValue(strategy = GenerationType.IDENTITY)
    private Long id;

    // ADMIN, PHARMACIEN, CAISSIER, CLIENT (sans le préfixe ROLE_, Spring l'ajoute)
    @Column(unique = true, nullable = false)
    private String nom;

    public Role(String nom) {
        this.nom = nom;
    }
}

Writing pharma_security/src/main/java/ma/ensa/pharmasecurity/model/Role.java


In [6]:
%%writefile pharma_security/src/main/java/ma/ensa/pharmasecurity/model/User.java

package ma.ensa.pharmasecurity.model;

import jakarta.persistence.*;
import lombok.Getter;
import lombok.NoArgsConstructor;
import lombok.Setter;

import java.util.HashSet;
import java.util.Set;

@Entity
@Table(name = "users")
@Getter
@Setter
@NoArgsConstructor
public class User {

    @Id
    @GeneratedValue(strategy = GenerationType.IDENTITY)
    private Long id;

    @Column(unique = true, nullable = false)
    private String username;

    // mot de passe chiffré avec BCrypt
    @Column(nullable = false)
    private String password;

    private boolean enabled = true;

    @ManyToMany(fetch = FetchType.EAGER)
    @JoinTable(
            name = "user_roles",
            joinColumns = @JoinColumn(name = "user_id"),
            inverseJoinColumns = @JoinColumn(name = "role_id")
    )
    private Set<Role> roles = new HashSet<>();
}

Writing pharma_security/src/main/java/ma/ensa/pharmasecurity/model/User.java


In [7]:
%%writefile pharma_security/src/main/java/ma/ensa/pharmasecurity/model/Medicament.java

package ma.ensa.pharmasecurity.model;

import jakarta.persistence.*;
import lombok.Getter;
import lombok.NoArgsConstructor;
import lombok.Setter;
import org.springframework.format.annotation.DateTimeFormat;

import java.math.BigDecimal;
import java.time.LocalDate;

@Entity
@Getter
@Setter
@NoArgsConstructor
public class Medicament {

    @Id
    @GeneratedValue(strategy = GenerationType.IDENTITY)
    private Long id;

    private String nom;

    @Column(length = 500)
    private String description;

    private BigDecimal prix;

    // quantité disponible en stock
    private int quantite;

    // les formulaires HTML envoient la date au format yyyy-MM-dd
    @DateTimeFormat(iso = DateTimeFormat.ISO.DATE)
    private LocalDate dateExpiration;
}

Writing pharma_security/src/main/java/ma/ensa/pharmasecurity/model/Medicament.java


In [8]:
%%writefile pharma_security/src/main/java/ma/ensa/pharmasecurity/model/Vente.java

package ma.ensa.pharmasecurity.model;

import jakarta.persistence.*;
import lombok.Getter;
import lombok.NoArgsConstructor;
import lombok.Setter;

import java.math.BigDecimal;
import java.time.LocalDateTime;
import java.time.format.DateTimeFormatter;

@Entity
@Getter
@Setter
@NoArgsConstructor
public class Vente {

    @Id
    @GeneratedValue(strategy = GenerationType.IDENTITY)
    private Long id;

    @ManyToOne(optional = false)
    @JoinColumn(name = "medicament_id")
    private Medicament medicament;

    private int quantite;
    private BigDecimal total;
    private LocalDateTime dateVente;

    // nom de l'utilisateur connecté qui a fait la vente
    private String vendeur;

    // pratique pour l'affichage dans la page
    public String getDateAffichee() {
        return dateVente == null ? "" : dateVente.format(DateTimeFormatter.ofPattern("dd/MM/yyyy HH:mm"));
    }
}

Writing pharma_security/src/main/java/ma/ensa/pharmasecurity/model/Vente.java


## 5. Repositories

In [9]:
%%writefile pharma_security/src/main/java/ma/ensa/pharmasecurity/repository/UserRepository.java

package ma.ensa.pharmasecurity.repository;

import ma.ensa.pharmasecurity.model.User;
import org.springframework.data.jpa.repository.JpaRepository;

import java.util.Optional;

public interface UserRepository extends JpaRepository<User, Long> {

    Optional<User> findByUsername(String username);

    boolean existsByUsername(String username);
}

Writing pharma_security/src/main/java/ma/ensa/pharmasecurity/repository/UserRepository.java


In [10]:
%%writefile pharma_security/src/main/java/ma/ensa/pharmasecurity/repository/RoleRepository.java

package ma.ensa.pharmasecurity.repository;

import ma.ensa.pharmasecurity.model.Role;
import org.springframework.data.jpa.repository.JpaRepository;

import java.util.Optional;

public interface RoleRepository extends JpaRepository<Role, Long> {

    Optional<Role> findByNom(String nom);
}

Writing pharma_security/src/main/java/ma/ensa/pharmasecurity/repository/RoleRepository.java


In [11]:
%%writefile pharma_security/src/main/java/ma/ensa/pharmasecurity/repository/MedicamentRepository.java

package ma.ensa.pharmasecurity.repository;

import ma.ensa.pharmasecurity.model.Medicament;
import org.springframework.data.jpa.repository.JpaRepository;

public interface MedicamentRepository extends JpaRepository<Medicament, Long> {
}

Writing pharma_security/src/main/java/ma/ensa/pharmasecurity/repository/MedicamentRepository.java


In [12]:
%%writefile pharma_security/src/main/java/ma/ensa/pharmasecurity/repository/VenteRepository.java

package ma.ensa.pharmasecurity.repository;

import ma.ensa.pharmasecurity.model.Vente;
import org.springframework.data.jpa.repository.JpaRepository;

import java.util.List;

public interface VenteRepository extends JpaRepository<Vente, Long> {

    List<Vente> findTop20ByOrderByDateVenteDesc();
}

Writing pharma_security/src/main/java/ma/ensa/pharmasecurity/repository/VenteRepository.java


## 6. Sécurité

**`CustomUserDetailsService`** : Spring Security l'appelle au moment du login. Elle charge l'utilisateur dans MySQL et transforme ses rôles en autorités (`ROLE_ADMIN`...).

**`SecurityConfig`** : règles d'accès par URL, page de login personnalisée, déconnexion, page `access-denied`. Les règles sont évaluées **dans l'ordre**, donc les plus précises sont placées avant les plus générales :

| URL | Rôles autorisés |
|---|---|
| `/login` | tout le monde |
| `/admin/**` | ADMIN |
| `/medicaments/nouveau`, `/medicaments/ajouter`, `/medicaments/*/supprimer` | ADMIN |
| `/medicaments/*/stock` | ADMIN, PHARMACIEN |
| `/medicaments` (consultation) | ADMIN, PHARMACIEN, CLIENT |
| `/ventes/**` | ADMIN, PHARMACIEN, CAISSIER |
| le reste | utilisateur connecté |

Le CSRF reste activé (par défaut). Les formulaires Thymeleaf avec `th:action` ajoutent le jeton automatiquement.

**`InitData`** : au premier démarrage, crée les 4 rôles, les comptes de test et quelques médicaments. Les mots de passe sont chiffrés avant d'être enregistrés.

In [13]:
%%writefile pharma_security/src/main/java/ma/ensa/pharmasecurity/service/CustomUserDetailsService.java

package ma.ensa.pharmasecurity.service;

import lombok.RequiredArgsConstructor;
import ma.ensa.pharmasecurity.model.Role;
import ma.ensa.pharmasecurity.repository.UserRepository;
import org.springframework.security.core.userdetails.UserDetails;
import org.springframework.security.core.userdetails.UserDetailsService;
import org.springframework.security.core.userdetails.UsernameNotFoundException;
import org.springframework.stereotype.Service;

@Service
@RequiredArgsConstructor
public class CustomUserDetailsService implements UserDetailsService {

    private final UserRepository userRepository;

    @Override
    public UserDetails loadUserByUsername(String username) throws UsernameNotFoundException {
        ma.ensa.pharmasecurity.model.User user = userRepository.findByUsername(username)
                .orElseThrow(() -> new UsernameNotFoundException("Utilisateur introuvable : " + username));

        String[] roles = user.getRoles().stream()
                .map(Role::getNom)
                .toArray(String[]::new);

        // roles(...) ajoute automatiquement le préfixe ROLE_
        return org.springframework.security.core.userdetails.User
                .withUsername(user.getUsername())
                .password(user.getPassword())
                .roles(roles)
                .disabled(!user.isEnabled())
                .build();
    }
}

Writing pharma_security/src/main/java/ma/ensa/pharmasecurity/service/CustomUserDetailsService.java


In [14]:
%%writefile pharma_security/src/main/java/ma/ensa/pharmasecurity/config/SecurityConfig.java

package ma.ensa.pharmasecurity.config;

import org.springframework.context.annotation.Bean;
import org.springframework.context.annotation.Configuration;
import org.springframework.security.config.annotation.web.builders.HttpSecurity;
import org.springframework.security.config.annotation.web.configuration.EnableWebSecurity;
import org.springframework.security.crypto.bcrypt.BCryptPasswordEncoder;
import org.springframework.security.crypto.password.PasswordEncoder;
import org.springframework.security.web.SecurityFilterChain;

@Configuration
@EnableWebSecurity
public class SecurityConfig {

    @Bean
    public PasswordEncoder passwordEncoder() {
        return new BCryptPasswordEncoder();
    }

    @Bean
    public SecurityFilterChain filterChain(HttpSecurity http) throws Exception {
        http
            .authorizeHttpRequests(auth -> auth
                .requestMatchers("/login", "/error").permitAll()
                .requestMatchers("/admin/**").hasRole("ADMIN")
                .requestMatchers("/medicaments/nouveau", "/medicaments/ajouter", "/medicaments/*/supprimer").hasRole("ADMIN")
                .requestMatchers("/medicaments/*/stock").hasAnyRole("ADMIN", "PHARMACIEN")
                .requestMatchers("/medicaments").hasAnyRole("ADMIN", "PHARMACIEN", "CLIENT")
                .requestMatchers("/ventes/**").hasAnyRole("ADMIN", "PHARMACIEN", "CAISSIER")
                .anyRequest().authenticated()
            )
            .formLogin(form -> form
                .loginPage("/login")
                .defaultSuccessUrl("/home", true)
                .permitAll()
            )
            .logout(logout -> logout
                .logoutSuccessUrl("/login?logout")
                .permitAll()
            )
            .exceptionHandling(ex -> ex
                .accessDeniedPage("/access-denied")
            );

        return http.build();
    }
}

Writing pharma_security/src/main/java/ma/ensa/pharmasecurity/config/SecurityConfig.java


In [15]:
%%writefile pharma_security/src/main/java/ma/ensa/pharmasecurity/service/InitData.java

package ma.ensa.pharmasecurity.service;

import lombok.RequiredArgsConstructor;
import ma.ensa.pharmasecurity.model.Medicament;
import ma.ensa.pharmasecurity.model.Role;
import ma.ensa.pharmasecurity.model.User;
import ma.ensa.pharmasecurity.repository.MedicamentRepository;
import ma.ensa.pharmasecurity.repository.RoleRepository;
import ma.ensa.pharmasecurity.repository.UserRepository;
import org.springframework.boot.CommandLineRunner;
import org.springframework.security.crypto.password.PasswordEncoder;
import org.springframework.stereotype.Component;

import java.math.BigDecimal;
import java.time.LocalDate;
import java.util.List;

@Component
@RequiredArgsConstructor
public class InitData implements CommandLineRunner {

    private final RoleRepository roleRepository;
    private final UserRepository userRepository;
    private final MedicamentRepository medicamentRepository;
    private final PasswordEncoder passwordEncoder;

    @Override
    public void run(String... args) {
        for (String nom : List.of("ADMIN", "PHARMACIEN", "CAISSIER", "CLIENT")) {
            if (roleRepository.findByNom(nom).isEmpty()) {
                roleRepository.save(new Role(nom));
            }
        }

        creerUtilisateur("admin", "admin123", "ADMIN");
        creerUtilisateur("pharma", "pharma123", "PHARMACIEN");
        creerUtilisateur("caisse", "caisse123", "CAISSIER");
        creerUtilisateur("client", "client123", "CLIENT");

        if (medicamentRepository.count() == 0) {
            medicamentRepository.save(medicament("Amoxicilline 500mg", "Antibiotique à large spectre.", "45.00", 120, "2027-03-10"));
            medicamentRepository.save(medicament("Doliprane 500mg", "Antalgique et antipyrétique.", "25.50", 200, "2027-11-01"));
            medicamentRepository.save(medicament("Vitamine C 1000mg", "Complément pour renforcer les défenses.", "30.00", 80, "2027-06-15"));
            medicamentRepository.save(medicament("Bétadine Solution 10%", "Désinfectant pour usage externe.", "18.00", 60, "2027-01-30"));
            medicamentRepository.save(medicament("Nurofen 400mg", "Soulage la douleur et la fièvre.", "42.00", 90, "2027-02-20"));
        }
    }

    private void creerUtilisateur(String username, String motDePasse, String nomRole) {
        if (userRepository.existsByUsername(username)) {
            return;
        }
        Role role = roleRepository.findByNom(nomRole).orElseThrow();
        User user = new User();
        user.setUsername(username);
        user.setPassword(passwordEncoder.encode(motDePasse)); // jamais en clair
        user.getRoles().add(role);
        userRepository.save(user);
    }

    private Medicament medicament(String nom, String description, String prix, int quantite, String expiration) {
        Medicament m = new Medicament();
        m.setNom(nom);
        m.setDescription(description);
        m.setPrix(new BigDecimal(prix));
        m.setQuantite(quantite);
        m.setDateExpiration(LocalDate.parse(expiration));
        return m;
    }
}

Writing pharma_security/src/main/java/ma/ensa/pharmasecurity/service/InitData.java


## 7. Contrôleurs et service de vente

Détail important : la page `/access-denied` est mappée avec `@RequestMapping` (toutes les méthodes HTTP) et pas seulement `@GetMapping`. Quand un `POST` est refusé, Spring Security fait un *forward* en gardant la méthode POST : avec `@GetMapping` on obtiendrait une erreur 405 au lieu de la page 403.

`VenteService.vendre(...)` est `@Transactional` : la vérification du stock, la baisse de la quantité et l'enregistrement de la vente se font ensemble (tout passe ou rien).

In [16]:
%%writefile pharma_security/src/main/java/ma/ensa/pharmasecurity/controller/LoginController.java

package ma.ensa.pharmasecurity.controller;

import org.springframework.http.HttpStatus;
import org.springframework.stereotype.Controller;
import org.springframework.web.bind.annotation.GetMapping;
import org.springframework.web.bind.annotation.RequestMapping;
import org.springframework.web.bind.annotation.ResponseStatus;

@Controller
public class LoginController {

    @GetMapping("/")
    public String racine() {
        return "redirect:/home";
    }

    @GetMapping("/login")
    public String login() {
        return "login";
    }

    @GetMapping("/home")
    public String home() {
        return "home";
    }

    // toutes les méthodes HTTP (voir la remarque plus haut)
    @RequestMapping("/access-denied")
    @ResponseStatus(HttpStatus.FORBIDDEN)
    public String accessDenied() {
        return "access-denied";
    }
}

Writing pharma_security/src/main/java/ma/ensa/pharmasecurity/controller/LoginController.java


In [17]:
%%writefile pharma_security/src/main/java/ma/ensa/pharmasecurity/controller/MedicamentController.java

package ma.ensa.pharmasecurity.controller;

import lombok.RequiredArgsConstructor;
import ma.ensa.pharmasecurity.model.Medicament;
import ma.ensa.pharmasecurity.repository.MedicamentRepository;
import org.springframework.dao.DataIntegrityViolationException;
import org.springframework.data.domain.Sort;
import org.springframework.http.HttpStatus;
import org.springframework.stereotype.Controller;
import org.springframework.ui.Model;
import org.springframework.web.bind.annotation.*;
import org.springframework.web.server.ResponseStatusException;
import org.springframework.web.servlet.mvc.support.RedirectAttributes;

@Controller
@RequestMapping("/medicaments")
@RequiredArgsConstructor
public class MedicamentController {

    private final MedicamentRepository medicamentRepository;

    // consultation : ADMIN, PHARMACIEN, CLIENT
    @GetMapping
    public String liste(Model model) {
        model.addAttribute("medicaments", medicamentRepository.findAll(Sort.by("nom")));
        return "medicaments";
    }

    // ajout : ADMIN seulement
    @GetMapping("/nouveau")
    public String nouveau() {
        return "medicament-form";
    }

    @PostMapping("/ajouter")
    public String ajouter(@ModelAttribute Medicament medicament, RedirectAttributes ra) {
        medicament.setId(null);
        medicamentRepository.save(medicament);
        ra.addFlashAttribute("message", "Médicament ajouté : " + medicament.getNom());
        return "redirect:/medicaments";
    }

    // modification du stock : ADMIN et PHARMACIEN
    @GetMapping("/{id}/stock")
    public String stockForm(@PathVariable Long id, Model model) {
        Medicament medicament = medicamentRepository.findById(id)
                .orElseThrow(() -> new ResponseStatusException(HttpStatus.NOT_FOUND, "Médicament introuvable"));
        model.addAttribute("medicament", medicament);
        return "stock-form";
    }

    @PostMapping("/{id}/stock")
    public String modifierStock(@PathVariable Long id, @RequestParam int quantite, RedirectAttributes ra) {
        Medicament medicament = medicamentRepository.findById(id)
                .orElseThrow(() -> new ResponseStatusException(HttpStatus.NOT_FOUND, "Médicament introuvable"));
        if (quantite < 0) {
            ra.addFlashAttribute("erreur", "Le stock ne peut pas être négatif");
            return "redirect:/medicaments";
        }
        medicament.setQuantite(quantite);
        medicamentRepository.save(medicament);
        ra.addFlashAttribute("message", "Stock mis à jour : " + medicament.getNom() + " (" + quantite + ")");
        return "redirect:/medicaments";
    }

    // suppression : ADMIN seulement
    @PostMapping("/{id}/supprimer")
    public String supprimer(@PathVariable Long id, RedirectAttributes ra) {
        try {
            medicamentRepository.deleteById(id);
            ra.addFlashAttribute("message", "Médicament supprimé");
        } catch (DataIntegrityViolationException e) {
            // il existe déjà des ventes pour ce médicament
            ra.addFlashAttribute("erreur", "Suppression impossible : des ventes sont liées à ce médicament");
        }
        return "redirect:/medicaments";
    }
}

Writing pharma_security/src/main/java/ma/ensa/pharmasecurity/controller/MedicamentController.java


In [18]:
%%writefile pharma_security/src/main/java/ma/ensa/pharmasecurity/controller/AdminController.java

package ma.ensa.pharmasecurity.controller;

import lombok.RequiredArgsConstructor;
import ma.ensa.pharmasecurity.model.Role;
import ma.ensa.pharmasecurity.model.User;
import ma.ensa.pharmasecurity.repository.RoleRepository;
import ma.ensa.pharmasecurity.repository.UserRepository;
import org.springframework.security.crypto.password.PasswordEncoder;
import org.springframework.stereotype.Controller;
import org.springframework.ui.Model;
import org.springframework.web.bind.annotation.*;
import org.springframework.web.servlet.mvc.support.RedirectAttributes;

import java.util.Optional;

@Controller
@RequestMapping("/admin")
@RequiredArgsConstructor
public class AdminController {

    private final UserRepository userRepository;
    private final RoleRepository roleRepository;
    private final PasswordEncoder passwordEncoder;

    @GetMapping
    public String admin(Model model) {
        model.addAttribute("users", userRepository.findAll());
        model.addAttribute("roles", roleRepository.findAll());
        return "admin";
    }

    @PostMapping("/users/ajouter")
    public String ajouter(@RequestParam String username,
                          @RequestParam String password,
                          @RequestParam String role,
                          RedirectAttributes ra) {
        String nom = username.trim();
        if (nom.isEmpty() || password.isBlank()) {
            ra.addFlashAttribute("erreur", "Nom d'utilisateur et mot de passe obligatoires");
            return "redirect:/admin";
        }
        if (userRepository.existsByUsername(nom)) {
            ra.addFlashAttribute("erreur", "Cet utilisateur existe déjà : " + nom);
            return "redirect:/admin";
        }
        Optional<Role> roleTrouve = roleRepository.findByNom(role);
        if (roleTrouve.isEmpty()) {
            ra.addFlashAttribute("erreur", "Rôle inconnu : " + role);
            return "redirect:/admin";
        }

        User user = new User();
        user.setUsername(nom);
        user.setPassword(passwordEncoder.encode(password));
        user.getRoles().add(roleTrouve.get());
        userRepository.save(user);

        ra.addFlashAttribute("message", "Utilisateur créé : " + nom);
        return "redirect:/admin";
    }

    @PostMapping("/users/{id}/supprimer")
    public String supprimer(@PathVariable Long id, RedirectAttributes ra) {
        Optional<User> user = userRepository.findById(id);
        if (user.isEmpty()) {
            ra.addFlashAttribute("erreur", "Utilisateur introuvable");
        } else if ("admin".equals(user.get().getUsername())) {
            ra.addFlashAttribute("erreur", "Le compte admin ne peut pas être supprimé");
        } else {
            userRepository.delete(user.get());
            ra.addFlashAttribute("message", "Utilisateur supprimé : " + user.get().getUsername());
        }
        return "redirect:/admin";
    }
}

Writing pharma_security/src/main/java/ma/ensa/pharmasecurity/controller/AdminController.java


In [19]:
%%writefile pharma_security/src/main/java/ma/ensa/pharmasecurity/service/VenteService.java

package ma.ensa.pharmasecurity.service;

import lombok.RequiredArgsConstructor;
import ma.ensa.pharmasecurity.model.Medicament;
import ma.ensa.pharmasecurity.model.Vente;
import ma.ensa.pharmasecurity.repository.MedicamentRepository;
import ma.ensa.pharmasecurity.repository.VenteRepository;
import org.springframework.stereotype.Service;
import org.springframework.transaction.annotation.Transactional;

import java.math.BigDecimal;
import java.time.LocalDateTime;

@Service
@RequiredArgsConstructor
public class VenteService {

    private final MedicamentRepository medicamentRepository;
    private final VenteRepository venteRepository;

    @Transactional
    public Vente vendre(Long medicamentId, int quantite, String vendeur) {
        if (quantite <= 0) {
            throw new IllegalArgumentException("La quantité doit être positive");
        }
        Medicament medicament = medicamentRepository.findById(medicamentId)
                .orElseThrow(() -> new IllegalArgumentException("Médicament introuvable"));
        if (medicament.getQuantite() < quantite) {
            throw new IllegalArgumentException("Stock insuffisant (" + medicament.getQuantite() + " disponible)");
        }

        // le médicament est géré par JPA : la baisse du stock est enregistrée à la fin de la transaction
        medicament.setQuantite(medicament.getQuantite() - quantite);

        BigDecimal prix = medicament.getPrix() != null ? medicament.getPrix() : BigDecimal.ZERO;

        Vente vente = new Vente();
        vente.setMedicament(medicament);
        vente.setQuantite(quantite);
        vente.setTotal(prix.multiply(BigDecimal.valueOf(quantite)));
        vente.setDateVente(LocalDateTime.now());
        vente.setVendeur(vendeur);
        return venteRepository.save(vente);
    }
}

Writing pharma_security/src/main/java/ma/ensa/pharmasecurity/service/VenteService.java


In [20]:
%%writefile pharma_security/src/main/java/ma/ensa/pharmasecurity/controller/VenteController.java

package ma.ensa.pharmasecurity.controller;

import lombok.RequiredArgsConstructor;
import ma.ensa.pharmasecurity.model.Vente;
import ma.ensa.pharmasecurity.repository.MedicamentRepository;
import ma.ensa.pharmasecurity.repository.VenteRepository;
import ma.ensa.pharmasecurity.service.VenteService;
import org.springframework.data.domain.Sort;
import org.springframework.security.core.Authentication;
import org.springframework.stereotype.Controller;
import org.springframework.ui.Model;
import org.springframework.web.bind.annotation.*;
import org.springframework.web.servlet.mvc.support.RedirectAttributes;

@Controller
@RequestMapping("/ventes")
@RequiredArgsConstructor
public class VenteController {

    private final VenteService venteService;
    private final VenteRepository venteRepository;
    private final MedicamentRepository medicamentRepository;

    @GetMapping
    public String page(Model model) {
        model.addAttribute("medicaments", medicamentRepository.findAll(Sort.by("nom")));
        model.addAttribute("ventes", venteRepository.findTop20ByOrderByDateVenteDesc());
        return "ventes";
    }

    @PostMapping("/enregistrer")
    public String enregistrer(@RequestParam Long medicamentId,
                              @RequestParam int quantite,
                              Authentication authentication,
                              RedirectAttributes ra) {
        try {
            Vente vente = venteService.vendre(medicamentId, quantite, authentication.getName());
            ra.addFlashAttribute("message", "Vente enregistrée : " + vente.getQuantite()
                    + " x " + vente.getMedicament().getNom() + " (" + vente.getTotal() + " DH)");
        } catch (IllegalArgumentException e) {
            ra.addFlashAttribute("erreur", e.getMessage());
        }
        return "redirect:/ventes";
    }
}

Writing pharma_security/src/main/java/ma/ensa/pharmasecurity/controller/VenteController.java


## 8. Pages Thymeleaf

Le module `thymeleaf-extras-springsecurity6` permet de cacher des éléments selon le rôle :
`sec:authorize="hasRole('ADMIN')"` (afficher seulement si ADMIN) et `sec:authentication="name"` (nom de l'utilisateur connecté). Attention : cacher un bouton ne protège pas l'URL, la vraie protection est dans `SecurityConfig`.

In [21]:
%%writefile pharma_security/src/main/resources/templates/layout.html

<!DOCTYPE html>
<html xmlns:th="http://www.thymeleaf.org"
      xmlns:sec="http://www.thymeleaf.org/extras/spring-security">
<head th:fragment="head(titre)">
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <title th:text="${titre} + ' - Pharmacie'">Pharmacie</title>
    <style>
        body { font-family: Arial, sans-serif; margin: 0; background: #f4f6f8; color: #222; }
        nav { background: #1f6f5c; padding: 12px 24px; display: flex; align-items: center; gap: 18px; }
        nav a { color: white; text-decoration: none; font-weight: bold; }
        nav .spacer { flex: 1; }
        nav .user { color: #d9efe9; }
        nav form { margin: 0; }
        nav button { background: #174f42; color: white; border: 0; padding: 6px 12px; border-radius: 4px; cursor: pointer; }
        main { padding: 24px; max-width: 1000px; margin: 0 auto; }
        table { border-collapse: collapse; width: 100%; background: white; margin-top: 12px; }
        th, td { border: 1px solid #ddd; padding: 8px 10px; text-align: left; }
        th { background: #e3efec; }
        input, select { padding: 7px; margin: 4px 6px 4px 0; }
        label { display: block; margin-top: 10px; font-weight: bold; }
        button.action, .btn { background: #1f6f5c; color: white; border: 0; padding: 7px 14px; border-radius: 4px; cursor: pointer; text-decoration: none; display: inline-block; }
        button.danger { background: #c0392b; color: white; border: 0; padding: 5px 10px; border-radius: 4px; cursor: pointer; }
        .ok { background: #d5f0e3; color: #145a32; padding: 8px 12px; border-radius: 4px; }
        .err { background: #f9d6d5; color: #922b21; padding: 8px 12px; border-radius: 4px; }
        .tag { background: #d6eaf8; border-radius: 10px; padding: 2px 8px; margin-right: 4px; font-size: 12px; }
        .card { background: white; border-radius: 6px; padding: 14px 18px; margin: 10px 0; box-shadow: 0 1px 3px rgba(0,0,0,.12); }
        .box { max-width: 360px; margin: 80px auto; background: white; padding: 24px; border-radius: 6px; box-shadow: 0 1px 6px rgba(0,0,0,.2); }
        .box input { width: 100%; box-sizing: border-box; }
    </style>
</head>
<body>
<nav th:fragment="nav">
    <a th:href="@{/home}">Accueil</a>
    <a th:href="@{/medicaments}" sec:authorize="hasAnyRole('ADMIN','PHARMACIEN','CLIENT')">Médicaments</a>
    <a th:href="@{/ventes}" sec:authorize="hasAnyRole('ADMIN','PHARMACIEN','CAISSIER')">Ventes</a>
    <a th:href="@{/admin}" sec:authorize="hasRole('ADMIN')">Administration</a>
    <span class="spacer"></span>
    <span class="user">Connecté : <b sec:authentication="name">user</b></span>
    <form th:action="@{/logout}" method="post">
        <button type="submit">Déconnexion</button>
    </form>
</nav>
<div th:fragment="messages">
    <p class="ok" th:if="${message}" th:text="${message}"></p>
    <p class="err" th:if="${erreur}" th:text="${erreur}"></p>
</div>
</body>
</html>

Writing pharma_security/src/main/resources/templates/layout.html


In [22]:
%%writefile pharma_security/src/main/resources/templates/login.html

<!DOCTYPE html>
<html xmlns:th="http://www.thymeleaf.org">
<head th:replace="~{layout :: head('Connexion')}"></head>
<body>
<div class="box">
    <h2>Gestion de pharmacie</h2>
    <p class="err" th:if="${param.error}">Nom d'utilisateur ou mot de passe incorrect.</p>
    <p class="ok" th:if="${param.logout}">Vous avez été déconnecté.</p>
    <form th:action="@{/login}" method="post">
        <label for="username">Nom d'utilisateur</label>
        <input type="text" id="username" name="username" required autofocus>
        <label for="password">Mot de passe</label>
        <input type="password" id="password" name="password" required>
        <p><button type="submit" class="action">Se connecter</button></p>
    </form>
</div>
</body>
</html>

Writing pharma_security/src/main/resources/templates/login.html


In [23]:
%%writefile pharma_security/src/main/resources/templates/home.html

<!DOCTYPE html>
<html xmlns:th="http://www.thymeleaf.org"
      xmlns:sec="http://www.thymeleaf.org/extras/spring-security">
<head th:replace="~{layout :: head('Accueil')}"></head>
<body>
<nav th:replace="~{layout :: nav}"></nav>
<main>
    <h1>Bienvenue <span sec:authentication="name">user</span></h1>
    <p>Vos rôles : <b sec:authentication="principal.authorities">roles</b></p>

    <div class="card" sec:authorize="hasAnyRole('ADMIN','PHARMACIEN','CLIENT')">
        <b>Médicaments</b><br>
        <span sec:authorize="hasRole('CLIENT')">Consultation des médicaments disponibles.</span>
        <span sec:authorize="hasRole('PHARMACIEN')">Consultation et modification des stocks.</span>
        <span sec:authorize="hasRole('ADMIN')">Gestion complète : ajout, stock, suppression.</span><br>
        <a th:href="@{/medicaments}">Ouvrir</a>
    </div>
    <div class="card" sec:authorize="hasAnyRole('ADMIN','PHARMACIEN','CAISSIER')">
        <b>Ventes</b><br>Enregistrer une vente.<br>
        <a th:href="@{/ventes}">Ouvrir</a>
    </div>
    <div class="card" sec:authorize="hasRole('ADMIN')">
        <b>Administration</b><br>Gérer les utilisateurs et leurs rôles.<br>
        <a th:href="@{/admin}">Ouvrir</a>
    </div>
</main>
</body>
</html>

Writing pharma_security/src/main/resources/templates/home.html


In [24]:
%%writefile pharma_security/src/main/resources/templates/medicaments.html

<!DOCTYPE html>
<html xmlns:th="http://www.thymeleaf.org"
      xmlns:sec="http://www.thymeleaf.org/extras/spring-security">
<head th:replace="~{layout :: head('Médicaments')}"></head>
<body>
<nav th:replace="~{layout :: nav}"></nav>
<main>
    <h1>Médicaments</h1>
    <div th:replace="~{layout :: messages}"></div>
    <a class="btn" th:href="@{/medicaments/nouveau}" sec:authorize="hasRole('ADMIN')">+ Nouveau médicament</a>
    <table>
        <thead>
        <tr><th>Nom</th><th>Description</th><th>Prix (DH)</th><th>Stock</th><th>Expiration</th>
            <th sec:authorize="hasAnyRole('ADMIN','PHARMACIEN')">Actions</th></tr>
        </thead>
        <tbody>
        <tr th:each="m : ${medicaments}" th:attr="data-id=${m.id},data-nom=${m.nom}">
            <td th:text="${m.nom}"></td>
            <td th:text="${m.description}"></td>
            <td th:text="${m.prix}"></td>
            <td class="qte" th:text="${m.quantite}"></td>
            <td th:text="${m.dateExpiration}"></td>
            <td sec:authorize="hasAnyRole('ADMIN','PHARMACIEN')">
                <a th:href="@{/medicaments/{id}/stock(id=${m.id})}">Modifier le stock</a>
                <form sec:authorize="hasRole('ADMIN')" th:action="@{/medicaments/{id}/supprimer(id=${m.id})}"
                      method="post" style="display:inline">
                    <button type="submit" class="danger">Supprimer</button>
                </form>
            </td>
        </tr>
        </tbody>
    </table>
</main>
</body>
</html>

Writing pharma_security/src/main/resources/templates/medicaments.html


In [25]:
%%writefile pharma_security/src/main/resources/templates/medicament-form.html

<!DOCTYPE html>
<html xmlns:th="http://www.thymeleaf.org">
<head th:replace="~{layout :: head('Nouveau médicament')}"></head>
<body>
<nav th:replace="~{layout :: nav}"></nav>
<main>
    <h1>Nouveau médicament</h1>
    <form th:action="@{/medicaments/ajouter}" method="post">
        <label>Nom</label>
        <input type="text" name="nom" required>
        <label>Description</label>
        <input type="text" name="description" size="50">
        <label>Prix (DH)</label>
        <input type="number" name="prix" step="0.01" min="0" required>
        <label>Quantité en stock</label>
        <input type="number" name="quantite" min="0" value="0" required>
        <label>Date d'expiration</label>
        <input type="date" name="dateExpiration">
        <p><button type="submit" class="action">Enregistrer</button>
           <a th:href="@{/medicaments}">Annuler</a></p>
    </form>
</main>
</body>
</html>

Writing pharma_security/src/main/resources/templates/medicament-form.html


In [26]:
%%writefile pharma_security/src/main/resources/templates/stock-form.html

<!DOCTYPE html>
<html xmlns:th="http://www.thymeleaf.org">
<head th:replace="~{layout :: head('Modifier le stock')}"></head>
<body>
<nav th:replace="~{layout :: nav}"></nav>
<main>
    <h1>Modifier le stock</h1>
    <p><b th:text="${medicament.nom}">Médicament</b> : stock actuel
       <b th:text="${medicament.quantite}">0</b></p>
    <form th:action="@{/medicaments/{id}/stock(id=${medicament.id})}" method="post">
        <label>Nouvelle quantité</label>
        <input type="number" name="quantite" min="0" th:value="${medicament.quantite}" required>
        <p><button type="submit" class="action">Mettre à jour</button>
           <a th:href="@{/medicaments}">Annuler</a></p>
    </form>
</main>
</body>
</html>

Writing pharma_security/src/main/resources/templates/stock-form.html


In [27]:
%%writefile pharma_security/src/main/resources/templates/ventes.html

<!DOCTYPE html>
<html xmlns:th="http://www.thymeleaf.org">
<head th:replace="~{layout :: head('Ventes')}"></head>
<body>
<nav th:replace="~{layout :: nav}"></nav>
<main>
    <h1>Enregistrer une vente</h1>
    <div th:replace="~{layout :: messages}"></div>
    <form th:action="@{/ventes/enregistrer}" method="post">
        <select name="medicamentId" required>
            <option th:each="m : ${medicaments}" th:value="${m.id}"
                    th:text="${m.nom} + ' (stock : ' + ${m.quantite} + ')'"></option>
        </select>
        <input type="number" name="quantite" min="1" value="1" required>
        <button type="submit" class="action">Vendre</button>
    </form>

    <h2>Dernières ventes</h2>
    <table>
        <thead><tr><th>Date</th><th>Médicament</th><th>Quantité</th><th>Total (DH)</th><th>Vendeur</th></tr></thead>
        <tbody>
        <tr th:each="v : ${ventes}">
            <td th:text="${v.dateAffichee}"></td>
            <td th:text="${v.medicament.nom}"></td>
            <td th:text="${v.quantite}"></td>
            <td th:text="${v.total}"></td>
            <td th:text="${v.vendeur}"></td>
        </tr>
        </tbody>
    </table>
</main>
</body>
</html>

Writing pharma_security/src/main/resources/templates/ventes.html


In [28]:
%%writefile pharma_security/src/main/resources/templates/admin.html

<!DOCTYPE html>
<html xmlns:th="http://www.thymeleaf.org">
<head th:replace="~{layout :: head('Administration')}"></head>
<body>
<nav th:replace="~{layout :: nav}"></nav>
<main>
    <h1>Gestion des utilisateurs</h1>
    <div th:replace="~{layout :: messages}"></div>

    <form th:action="@{/admin/users/ajouter}" method="post">
        <input type="text" name="username" placeholder="Nom d'utilisateur" required>
        <input type="password" name="password" placeholder="Mot de passe" required>
        <select name="role">
            <option th:each="r : ${roles}" th:value="${r.nom}" th:text="${r.nom}"></option>
        </select>
        <button type="submit" class="action">Ajouter</button>
    </form>

    <table>
        <thead><tr><th>Id</th><th>Utilisateur</th><th>Rôles</th><th>Actions</th></tr></thead>
        <tbody>
        <tr th:each="u : ${users}" th:attr="data-id=${u.id},data-username=${u.username}">
            <td th:text="${u.id}"></td>
            <td th:text="${u.username}"></td>
            <td><span class="tag" th:each="r : ${u.roles}" th:text="${r.nom}"></span></td>
            <td>
                <form th:action="@{/admin/users/{id}/supprimer(id=${u.id})}" method="post">
                    <button type="submit" class="danger">Supprimer</button>
                </form>
            </td>
        </tr>
        </tbody>
    </table>
</main>
</body>
</html>

Writing pharma_security/src/main/resources/templates/admin.html


In [29]:
%%writefile pharma_security/src/main/resources/templates/access-denied.html

<!DOCTYPE html>
<html xmlns:th="http://www.thymeleaf.org">
<head th:replace="~{layout :: head('Accès refusé')}"></head>
<body>
<nav th:replace="~{layout :: nav}"></nav>
<main>
    <h1>Accès refusé (403)</h1>
    <p class="err">Votre rôle ne vous permet pas d'accéder à cette page.</p>
    <p><a th:href="@{/home}">Retour à l'accueil</a></p>
</main>
</body>
</html>

Writing pharma_security/src/main/resources/templates/access-denied.html


## 9. Lancer l'application

La cellule démarre `mvn spring-boot:run` en arrière-plan (logs dans `pharma_security/app.log`) et attend que la page `/login` réponde. Le premier démarrage peut durer 1 à 3 minutes (téléchargement des dépendances Maven).

In [30]:
import subprocess, time, requests, shutil

BASE_URL = "http://localhost:8081"

mvn = shutil.which("mvn")          # sous Windows ça trouve mvn.cmd
assert mvn, "Maven introuvable : vérifier le PATH puis redémarrer Jupyter"

log = open("pharma_security/app.log", "w")
serveur = subprocess.Popen([mvn, "-q", "spring-boot:run"], cwd="pharma_security",
                           stdout=log, stderr=subprocess.STDOUT)

demarre = False
for _ in range(120):                      # 120 x 3 s = 6 min max
    if serveur.poll() is not None:
        break                             # maven s'est arrêté : voir app.log
    try:
        if requests.get(BASE_URL + "/login", timeout=2).status_code == 200:
            demarre = True
            break
    except requests.exceptions.RequestException:
        pass
    time.sleep(3)

print("Application démarrée" if demarre else "Échec du démarrage, regarde pharma_security/app.log")

Application démarrée


## 10. Tests

### 10.1 Connexion

Pour se connecter par script, il faut d'abord lire le jeton CSRF caché dans la page de login, puis le renvoyer avec le formulaire. C'est exactement ce que fait le navigateur.

In [31]:
import re

def connexion(utilisateur, mot_de_passe):
    """Se connecte comme le ferait un navigateur. Retourne (session, succes)."""
    s = requests.Session()
    page = s.get(BASE_URL + "/login", timeout=15)
    token = re.search(r'name="_csrf"\s+value="([^"]+)"', page.text).group(1)
    r = s.post(BASE_URL + "/login",
               data={"username": utilisateur, "password": mot_de_passe, "_csrf": token},
               timeout=15)
    return s, r.url.rstrip("/").endswith("/home")

def csrf(session):
    """Jeton CSRF de la session (présent dans le formulaire de déconnexion de /home)."""
    page = session.get(BASE_URL + "/home", timeout=15)
    return re.search(r'name="_csrf"\s+value="([^"]+)"', page.text).group(1)

def post(session, chemin, data=None):
    donnees = dict(data or {})
    donnees["_csrf"] = csrf(session)
    return session.post(BASE_URL + chemin, data=donnees, allow_redirects=True, timeout=15)

comptes = {"admin": "admin123", "pharma": "pharma123", "caisse": "caisse123", "client": "client123"}
sessions = {}
for nom, mdp in comptes.items():
    s, ok = connexion(nom, mdp)
    assert ok, "connexion impossible pour " + nom
    sessions[nom] = s
    print("[OK  ] connexion de", nom)

_, ok = connexion("admin", "mauvais_mot_de_passe")
assert not ok
print("[OK  ] mauvais mot de passe : refusé")

[OK  ] connexion de admin
[OK  ] connexion de pharma
[OK  ] connexion de caisse
[OK  ] connexion de client
[OK  ] mauvais mot de passe : refusé


### 10.2 Matrice des accès

Chaque ligne est une page, chaque colonne un utilisateur. Codes attendus : **200** (accès autorisé), **403** (connecté mais rôle insuffisant), **302** (pas connecté : redirection vers `/login`).

In [32]:
def trouver_id(session, page, cle, valeur):
    """Cherche dans le tableau d'une page la ligne dont data-<cle>=<valeur> et renvoie son data-id."""
    html = session.get(BASE_URL + page, timeout=15).text
    for tag in re.findall(r"<tr[^>]*>", html):
        if 'data-' + cle + '="' + valeur + '"' in tag:
            return int(re.search(r'data-id="(\d+)"', tag).group(1))
    return None

def statut(session, chemin):
    return session.get(BASE_URL + chemin, allow_redirects=False, timeout=15).status_code

id_doli = trouver_id(sessions["admin"], "/medicaments", "nom", "Doliprane 500mg")
sessions["anonyme"] = requests.Session()

pages = ["/home", "/medicaments", "/medicaments/nouveau",
         "/medicaments/" + str(id_doli) + "/stock", "/ventes", "/admin"]
libelles = ["/home", "/medicaments", "/medicaments/nouveau", "/medicaments/{id}/stock", "/ventes", "/admin"]
attendu = {
    "admin":   [200, 200, 200, 200, 200, 200],
    "pharma":  [200, 200, 403, 200, 200, 403],
    "caisse":  [200, 403, 403, 403, 200, 403],
    "client":  [200, 200, 403, 403, 403, 403],
    "anonyme": [302, 302, 302, 302, 302, 302],
}

erreurs = 0
print("page".ljust(26) + "".join(u.rjust(11) for u in attendu))
for i, page in enumerate(pages):
    ligne = libelles[i].ljust(26)
    for utilisateur in attendu:
        code_http = statut(sessions[utilisateur], page)
        bon = (code_http == attendu[utilisateur][i])
        erreurs += 0 if bon else 1
        ligne += (str(code_http) + (" ok" if bon else " KO")).rjust(11)
    print(ligne)

assert erreurs == 0, str(erreurs) + " accès ne correspondent pas aux règles"
print()
print("Tous les accès respectent les règles de sécurité")

page                            admin     pharma     caisse     client    anonyme
/home                          200 ok     200 ok     200 ok     200 ok     302 ok
/medicaments                   200 ok     200 ok     403 ok     200 ok     302 ok
/medicaments/nouveau           200 ok     403 ok     403 ok     403 ok     302 ok
/medicaments/{id}/stock        200 ok     200 ok     403 ok     403 ok     302 ok
/ventes                        200 ok     200 ok     200 ok     403 ok     302 ok
/admin                         200 ok     403 ok     403 ok     403 ok     302 ok

Tous les accès respectent les règles de sécurité


### 10.3 Tests fonctionnels

Ici on vérifie les actions elles-mêmes : une vente baisse le stock, un pharmacien modifie le stock, et les actions interdites renvoient bien 403 même avec un jeton CSRF valide (sinon on ne saurait pas si c'est le CSRF ou le rôle qui bloque).

In [33]:
admin, pharma, caisse, client = (sessions[k] for k in ["admin", "pharma", "caisse", "client"])

def stock(id_med):
    html = admin.get(BASE_URL + "/medicaments", timeout=15).text
    m = re.search(r'<tr[^>]*data-id="%d"[^>]*>.*?class="qte">(\d+)<' % id_med, html, re.S)
    return int(m.group(1))

# --- CAISSIER : enregistrer une vente ---
avant = stock(id_doli)
r = post(caisse, "/ventes/enregistrer", {"medicamentId": id_doli, "quantite": 2})
assert r.status_code == 200 and "Vente enregistrée" in r.text
assert stock(id_doli) == avant - 2
print("[OK  ] caissier : vente de 2 boîtes, stock", avant, "->", stock(id_doli))

# stock insuffisant : refusé, stock inchangé
avant = stock(id_doli)
r = post(caisse, "/ventes/enregistrer", {"medicamentId": id_doli, "quantite": 100000})
assert "Stock insuffisant" in r.text and stock(id_doli) == avant
print("[OK  ] caissier : vente trop grande refusée, stock inchangé")

# --- PHARMACIEN : modifier le stock ---
r = post(pharma, "/medicaments/" + str(id_doli) + "/stock", {"quantite": 100})
assert r.status_code == 200 and stock(id_doli) == 100
print("[OK  ] pharmacien : stock modifié, nouveau stock =", stock(id_doli))

# --- Actions interdites : 403 ---
interdits = [
    ("caissier ne peut pas ajouter un médicament", caisse, "/medicaments/ajouter", {"nom": "X", "prix": "1", "quantite": "1"}),
    ("client ne peut pas vendre", client, "/ventes/enregistrer", {"medicamentId": id_doli, "quantite": 1}),
    ("client ne peut pas modifier le stock", client, "/medicaments/" + str(id_doli) + "/stock", {"quantite": 5}),
    ("pharmacien ne peut pas supprimer", pharma, "/medicaments/" + str(id_doli) + "/supprimer", {}),
    ("pharmacien ne peut pas créer d'utilisateur", pharma, "/admin/users/ajouter", {"username": "x", "password": "x", "role": "ADMIN"}),
]
for libelle, session, chemin, data in interdits:
    r = post(session, chemin, data)
    assert r.status_code == 403, libelle + " : code " + str(r.status_code)
    print("[OK  ] 403 :", libelle)
assert stock(id_doli) == 100
print("      le stock n'a pas bougé :", stock(id_doli))

[OK  ] caissier : vente de 2 boîtes, stock 200 -> 198
[OK  ] caissier : vente trop grande refusée, stock inchangé
[OK  ] pharmacien : stock modifié, nouveau stock = 100
[OK  ] 403 : caissier ne peut pas ajouter un médicament
[OK  ] 403 : client ne peut pas vendre
[OK  ] 403 : client ne peut pas modifier le stock
[OK  ] 403 : pharmacien ne peut pas supprimer
[OK  ] 403 : pharmacien ne peut pas créer d'utilisateur
      le stock n'a pas bougé : 100


In [34]:
# --- ADMIN : gérer les médicaments ---
r = post(admin, "/medicaments/ajouter", {"nom": "Voltaren 50mg", "description": "Anti-inflammatoire",
                                          "prix": "58.90", "quantite": 40, "dateExpiration": "2027-12-15"})
assert r.status_code == 200 and "Médicament ajouté" in r.text
id_volta = trouver_id(admin, "/medicaments", "nom", "Voltaren 50mg")
assert id_volta is not None
print("[OK  ] admin : médicament ajouté (id", id_volta, ")")

# le client voit le nouveau médicament (lecture seule)
assert "Voltaren 50mg" in client.get(BASE_URL + "/medicaments", timeout=15).text
print("[OK  ] client : voit le nouveau médicament")

post(admin, "/medicaments/" + str(id_volta) + "/supprimer")
assert trouver_id(admin, "/medicaments", "nom", "Voltaren 50mg") is None
print("[OK  ] admin : médicament supprimé")

# --- ADMIN : gérer les utilisateurs ---
r = post(admin, "/admin/users/ajouter", {"username": "test_user", "password": "test123", "role": "CLIENT"})
assert "Utilisateur créé" in r.text
_, ok = connexion("test_user", "test123")
assert ok
print("[OK  ] admin : utilisateur test_user créé, il peut se connecter")

id_user = trouver_id(admin, "/admin", "username", "test_user")
post(admin, "/admin/users/" + str(id_user) + "/supprimer")
_, ok = connexion("test_user", "test123")
assert not ok
print("[OK  ] admin : utilisateur supprimé, connexion refusée")

# --- Déconnexion ---
r = post(client, "/logout")
assert r.url.endswith("/login?logout")
assert statut(client, "/home") == 302
print("[OK  ] déconnexion : /home redirige vers /login")
print()
print("Tous les tests fonctionnels sont passés")

[OK  ] admin : médicament ajouté (id 6 )
[OK  ] client : voit le nouveau médicament
[OK  ] admin : médicament supprimé
[OK  ] admin : utilisateur test_user créé, il peut se connecter
[OK  ] admin : utilisateur supprimé, connexion refusée
[OK  ] déconnexion : /home redirige vers /login

Tous les tests fonctionnels sont passés


### 10.4 Mots de passe chiffrés

Pour vérifier que les mots de passe ne sont pas en clair en base, on peut regarder la table dans MySQL (phpMyAdmin ou ligne de commande) :

```sql
SELECT username, password FROM pharma_security_db.users;
```

Les valeurs commencent par `$2a$10$...` : c'est le format BCrypt (un hachage à sens unique, on ne peut pas retrouver `admin123` à partir de lui).

## 11. Arrêter l'application

In [35]:
import os
if os.name == "nt":
    # sous Windows, terminate() ne tue pas le processus java lancé par maven
    subprocess.run(["taskkill", "/F", "/T", "/PID", str(serveur.pid)],
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
else:
    serveur.terminate()
try:
    serveur.wait(timeout=20)
except subprocess.TimeoutExpired:
    serveur.kill()
log.close()
print("Application arrêtée")

Application arrêtée


## Conclusion

- **Authentification** : formulaire de login, utilisateurs et rôles en base MySQL (`CustomUserDetailsService`), mots de passe chiffrés avec BCrypt.
- **Autorisation** : règles par URL dans `SecurityConfig` (ADMIN, PHARMACIEN, CAISSIER, CLIENT) et affichage adapté au rôle avec `sec:authorize`.
- **Protection** : CSRF activé, page 403 personnalisée, déconnexion en POST.
- **Tests** : la matrice 5 profils x 6 pages et les tests fonctionnels vérifient les règles, y compris les actions interdites (403 même avec un jeton CSRF valide).

**Pour publier sur GitHub :** ajouter un `.gitignore` avec `pharma_security/target/` et `pharma_security/app.log`, exécuter toutes les cellules, sauvegarder le notebook (les sorties sont conservées dans le fichier), puis l'envoyer sur le dépôt.